In [ ]:
import os
import time
import gradio as gr
from dotenv import load_dotenv
from typing import List
from abc import ABC, abstractmethod
from openai import OpenAI
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from pydantic import BaseModel
from IPython.display import display, HTML


In [ ]:
display(HTML(f"""
<h3>Reflection / Self-Critique Agentic AI Pattern</h3>
<p>
    This pattern enables an AI agent to evaluate its own outputs, identify weaknesses or inconsistencies, and iteratively refine results without immediate human prompting. It forms a closed-loop self-assessment mechanism, often leveraging internal reasoning, feedback from prior outputs, and contextual cues from the task environment.
    This also demonstrates basic Agent Orchestration.
</p></br>
<img src='../../images/agentic-reflection-pattern.png' width='90%' height='300px'/></br></b>
"""))

display(HTML("""
<p>
<h2>Example:- </h2>
Answer user queries based on profile present under (me) folder acting as the user defined in summary.txt
Evaluate the response and regenerate the response if not found satisfactory. Evaluation demonstrates usage of:
<font style='color:lightgreen'>Structured Output Parameters</font>. </br></br>

<b>Note:-</b>
</br></br>
This example demonstrates <font style='color:lightgreen'>Automous Behavior</font> of Agentic System, as it self evaluates the output and regenerates the response till 
satisfactory response is received. Once satisfactory response is received present the response to the user.

</br></br> Evaluation and Computation loop can be perceived as an equivalent of <font style='color:lightgreen'>Agent Orchestration</font>,
as unsatisfactory resonses are not presented to the user. Response are recomputed till we get a valid response.

 
</br></br>
As agentic evaluation in this example can happen any number of times till satisfactory response is received, execution of this Agentic process can lead to un-deterministic number of execution of Computation and Evaluation loop. 
Example can be extended with a <font style='color:lightgreen'>Policy</font> to control max number of times Computation and Evaluation loop is permitted for execution.
</p>
"""))

In [ ]:

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
openai_api_key_found = openai_api_key and openai_api_key.startswith('sk-proj-') and len(openai_api_key) > len('sk-proj-')

if openai_api_key_found:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set - please setup OPENAI_API_KEY in .env file.")
    raise Exception("Please set Open API Key to proceed.")

In [ ]:
PRINT_ENABLED = False

def printHtmlContentWithHeader(header, content, displayFlag = True):
    printLog = displayFlag
    if PRINT_ENABLED:
        printLog = PRINT_ENABLED and displayFlag
    
    if printLog:    
        display(HTML(f"""
        <h2>{header}</h2>
        <p>{content}</p>
        </br>
        """))

# Read me/User-Profile.pdf and extract user information from the document.
def readProfile():
    text = ""
    reader = PdfReader("../../me/user-profile.pdf")
    for page in reader.pages:
        t = page.extract_text()
        if t:
            text += t

    if len(text) > 0:
        # Clean up data
        text = text.strip().replace("\n", " ").replace("• • • • • • • • • • • • • • • \u200b \u200b \u200b\u200b \u200b \u200b", "")
    printHtmlContentWithHeader("User Profile Profile", text, False)
    return text

# Short summary about the user read from me/summary.txt
def readSummary():
    summaryText = ""
    with open("../../me/summary.txt", "r", encoding="utf-8") as f:
        summaryText = f.read()
    if len(summaryText) > 0:
        summaryText = summaryText.strip().replace("\n", " ")
    printHtmlContentWithHeader("Summary", summaryText, False)
    return summaryText

# Populate User Context data
USER_NAME = "ETHAN HUNT"
PROFILE_DATA = readProfile()
SUMMARY = readSummary()


In [ ]:
SUMMARY

In [ ]:
PROFILE_DATA

In [ ]:
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

class QueryResponse:
    rawResponse: any
    reply: str

    def __init__(self, resp, reply):
        self.rawResponse = resp
        self.reply = reply

class GuardEvaluationResponse:
    is_acceptable: bool
    feedback: str

    def __init__(self):
        self.is_acceptable = True
        self.feedback = ""
    
class ResponseEvaluatorBase(ABC):
    
    @abstractmethod
    def evaluate(self, msg, rep, hist) -> Evaluation:
        pass

class GuardEvaluatorBase(ABC):
    @abstractmethod
    def evaluate(self, msg) -> GuardEvaluationResponse:
        pass


In [ ]:
class InputMessageGuard(GuardEvaluatorBase):
    # Here we are comparing against static values but in real world scenarios we can restrict based on some configurations
    # or parameters that may be relevant to system under consideration.
    forbiddenKeyWords: List[str] = ['employee id', 'eid', 'nino', 'national insurance number']

    def __init__(self, exclusions: List[str]):
        if exclusions and len(exclusions) > 0:
            self.forbiddenKeyWords.clear();
            for item in exclusions:
                if (item):
                    self.forbiddenKeyWords.append(item.lower().strip())

    def evaluate(self, msg) -> GuardEvaluationResponse:
        query = ""
        if msg and len(msg) > 0:
            query = msg.lower()

        evaluation = GuardEvaluationResponse()
                
        if self.forbiddenKeyWords and len(self.forbiddenKeyWords):
            for item in self.forbiddenKeyWords:
                if query.find(item) > -1:
                    evaluation.is_acceptable = False
                    evaluation.feedback = f"Querying about {item} is not allowed. Please try with another query"
                    break

        return evaluation

In [ ]:
# User Profile Response Evaluator Agent uses Google Gemini
# https://ai.google.dev/gemini-api/docs/openai
class UserProfileAgentResponseEvaluator(ResponseEvaluatorBase):

    openai= OpenAI()

    #gemini= OpenAI(
    #    api_key=os.getenv("GOOGLE_API_KEY"), 
    #    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    #)

    def __init__(self, usrName: str,profileData: str, summaryText: str):
        self.user_name = usrName
        self.user_profile = profileData
        self.user_summary = summaryText

        self.system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
        You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
        The Agent is playing the role of {usrName} and is representing {usrName} on their website. \
        The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
        The Agent has been provided with context on {usrName} in the form of their summary and LinkedIn details. Here's the information:"

        self.system_prompt += f"\n\n## Summary:\n{summaryText}\n\n## LinkedIn Profile:\n{profileData}\n\n"
        self.system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."
                
    
    def getUserPrompt(self, rep, msg, hist) -> str:
        user_prompt = f"Here's the conversation between the User and the Agent: \n\n{hist}\n\n"
        user_prompt += f"Here's the latest message from the User: \n\n{msg}\n\n"
        user_prompt += f"Here's the latest response from the Agent: \n\n{rep}\n\n"
        user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
        user_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."
        
        # printHtmlContentWithHeader("User Message", msg)
        printHtmlContentWithHeader("Agent Response", rep)
        # printHtmlContentWithHeader("Evaluator User Prompt", user_prompt)
        return user_prompt

    def evaluate(self, msg, rep, hist) -> Evaluation:
        messages = [{"role": "system", "content": self.system_prompt}] + [{"role": "user", "content": self.getUserPrompt(rep, msg, hist)}]
        #response = self.gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
        response = self.openai.chat.completions.parse(model="gpt-4o-mini", messages=messages, response_format=Evaluation)
        return response.choices[0].message.parsed

    

In [ ]:

# UserProfile Agent will use gpt-4o-mini model for computing response to queries #
class UserProfileAgent:
    # Initalize OPEN AI
    openai= OpenAI()

    # Evaluator Agent: Responds if response is accepted in evaluation
    query_response_evaluator: ResponseEvaluatorBase

    # Input message evaluation GuardRails: Responds if input message query is allowed as per policy 
    # Note:- Guard Rails can be implemented as functions or can be agents themselves. Here we will use Agent as Guardrail.   
    entry_guard_rails: List[GuardEvaluatorBase] = []

    default_system_prompt: str = ""

    # Constructor method
    def __init__(self, 
    usrName: str, 
    profileData: str, 
    summaryText: str, 
    responseEvaluatorAgent: ResponseEvaluatorBase,
    inputGuardRails: List[GuardEvaluatorBase]):
        self.user_name = usrName
        self.user_profile = profileData
        self.user_summary = summaryText
        self.query_response_evaluator = responseEvaluatorAgent
        if inputGuardRails and len(inputGuardRails) > 0:
            self.entry_guard_rails = inputGuardRails

        # ******************* This is default System Prompt used for computing response to User Query ******************* #
        self.default_system_prompt = f"You are acting as {usrName}. You are answering questions on {usrName}'s website, \
        particularly questions related to {usrName}'s career, background, skills and experience. \
        Your responsibility is to represent {usrName} for interactions on the website as faithfully as possible. \
        You are given a summary of {usrName}'s background and LinkedIn profile which you can use to answer questions. \
        Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
        If you don't know the answer, say so."

        self.default_system_prompt += f"\n\n## Summary:\n{summaryText}\n\n## LinkedIn Profile:\n{profileData}\n\n"
        self.default_system_prompt += f"With this context, please chat with the user, always staying in character as {usrName}."
    

    # ******************* Method to reterive System Prompt. Note in our scenario we teweek it to simulate Un-Succssful response ******************* #
    def getSystemPrompt(self, rep = "", fdbk = "", reVal = False, simulateFail = False) -> str:
        systemPrompt = self.default_system_prompt

        if simulateFail == True:
            systemPrompt = self.default_system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
            it is mandatory that you respond only and entirely in pig latin"
        elif reVal == True:
            systemPrompt = f"""
            {systemPrompt} \n
            Previous answer rejected. You just tried to reply, but the quality control rejected your reply.
            Your attempted answer: {rep}
            Reason for rejection: {fdbk}
            """
        return systemPrompt

    # Get message to evaluate passing it systemPrompt, userPrompt and history if any.
    def getMessages(self, sysPrompt, usrPrompt, hist):
        messages = [{"role": "system", "content": sysPrompt}] + [{"role": "user", "content": usrPrompt}]
        if hist:
            messages = [{"role": "system", "content": sysPrompt}] + hist + [{"role": "user", "content": usrPrompt}]
        
        return messages

    # Use OPENAI and gpt-4o-mini to compute response to user query #
    def executeQuery(self, msgs) -> QueryResponse:
        queryResponse = self.openai.chat.completions.create(model="gpt-4o-mini", messages=msgs)
        replyContent = queryResponse.choices[0].message.content
        return QueryResponse(queryResponse, replyContent)

    
    def askQuery(self, message, rep, fdbk, hist) -> str:
        sysPrompt = ""
        finalResponse = ""
        guardResponse: GuardEvaluationResponse = GuardEvaluationResponse()

        # If input guards are configured run user query message through the input 
        if self.entry_guard_rails and len(self.entry_guard_rails) > 0:
            for entryGuard in self.entry_guard_rails:
                if entryGuard:
                    guardEvaluation = entryGuard.evaluate(message)
                    if guardEvaluation and guardEvaluation.is_acceptable == False:
                        guardResponse = guardEvaluation
                        break
                
        if guardResponse and guardResponse.is_acceptable == False:
                # In case of input guard failing incoming query return response from first failed guard to notify user #
                yield guardResponse.feedback    
        else:
            # Fake a failure if user asks about song, singer or music related queries
            if "singer" in message or "song" in message or "music" in message:
                sysPrompt = self.getSystemPrompt(simulateFail=True)
                # printHtmlContentWithHeader("Failure System Prompt", sysPrompt, True)
            else:
                sysPrompt = self.getSystemPrompt()
            
            # sysPrompt = self.getSystemPrompt(rep = rep, fdbk = fdbk, reVal = False)
            messages = self.getMessages(sysPrompt= sysPrompt, usrPrompt= message, hist= hist)
            queryResponse = self.executeQuery(messages)
            evaluatorResponse = self.query_response_evaluator.evaluate(rep= queryResponse.reply, msg=message, hist= messages[:1])
            print(f"Query Response Acceptable: {evaluatorResponse.is_acceptable}")

            if evaluatorResponse and evaluatorResponse.is_acceptable:
                finalResponse = queryResponse.reply
                yield finalResponse
            else:
                finalResponse = f"""Invalid Response Received: {queryResponse.reply}. Retrying ... """
                yield finalResponse;
                time.sleep(1) ## Introduce delay for simulation ##
                sysPrompt = self.getSystemPrompt(rep = queryResponse.reply, fdbk = evaluatorResponse.feedback, reVal = True, simulateFail= False)
                messages = self.getMessages(sysPrompt= sysPrompt, usrPrompt= message, hist= hist)
                queryResponse = self.executeQuery(messages)
                finalResponse = queryResponse.reply
                yield finalResponse       


In [ ]:

inputGuards: List[GuardEvaluatorBase] = []
inputGuards.append(InputMessageGuard([]))
userQueryResponseEvaluator = UserProfileAgentResponseEvaluator(USER_NAME, PROFILE_DATA, SUMMARY)

userQueryAgent = UserProfileAgent(USER_NAME, 
PROFILE_DATA, 
SUMMARY, 
inputGuardRails=inputGuards, 
responseEvaluatorAgent = userQueryResponseEvaluator)

gr.ChatInterface(fn = userQueryAgent.askQuery).launch()